## Installing Libraries

In [1]:
!pip install -q transformers peft datasets
!pip install -q langchain
!pip install -q langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


## Importing Libraries

In [2]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    default_data_collator,
    pipeline
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from langchain_core.prompts import PromptTemplate
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

## Load the Pre-trained Model and Tokenizer

In [3]:
#Set device to GPU if available, otherwise CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
model_name = "tiiuae/Falcon3-1B-Base"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/91.0 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

##Configure Low-Rank Adaptation (LoRA) for Parameter-Efficient Fine-Tuning (PEFT)

In [5]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["k_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

## Load the Dataset

In [6]:
qa_dataset = load_dataset("squad", split="train[:2000]")
print(qa_dataset)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 2000
})


In [7]:
qa_dataset[0]

{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

## Preprocess the Dataset

In [8]:
#Format as instruction-style QA
def format_example(example):
    question = example["question"].strip()
    context = example["context"].strip()

    # SQuAD answers is a dict with lists; use first answer
    if len(example["answers"]["text"]) > 0:
        answer = example["answers"]["text"][0].strip()
    else:
        answer = "No answer available."

    text = f"""### Context:
    {context}

    ### Question:
    {question}

    ### Answer:
    {answer}{tokenizer.eos_token}"""

    return {"text": text}

formatted_dataset = qa_dataset.map(format_example)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [9]:
# keep only text column
cols_to_remove = [c for c in formatted_dataset.column_names if c != "text"]
formatted_dataset = formatted_dataset.remove_columns(cols_to_remove)

print("Formatted example:")
print(formatted_dataset[0]["text"][:1000])


Formatted example:
### Context:
    Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.

    ### Question:
    To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

    ### Answer:
    Saint Bernadette Soubirous<|endoftext|>


In [10]:
def tokenize_function(examples):
    full_texts = examples["text"]

    prompts = []
    for text in full_texts:
        parts = text.split("### Answer:\n", 1)
        prompt = parts[0] + "### Answer:\n"
        prompts.append(prompt)

    tokenized_full = tokenizer(
        full_texts,
        truncation=True,
        max_length=384,
        padding="max_length",
    )

    tokenized_prompt = tokenizer(
        prompts,
        truncation=True,
        max_length=384,
        padding="max_length",
    )

    labels = []
    for input_ids, prompt_ids in zip(tokenized_full["input_ids"], tokenized_prompt["input_ids"]):
        prompt_len = sum(1 for t in prompt_ids if t != tokenizer.pad_token_id)
        label_ids = input_ids.copy()

        # mask prompt tokens
        for i in range(prompt_len):
            label_ids[i] = -100

        # mask padding tokens
        label_ids = [
            -100 if tok == tokenizer.pad_token_id else lab
            for tok, lab in zip(input_ids, label_ids)
        ]

        labels.append(label_ids)

    tokenized_full["labels"] = labels
    return tokenized_full

In [11]:
tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [12]:
# Remove empty rows just in case
tokenized_dataset = tokenized_dataset.filter(
    lambda x: len(x["input_ids"]) > 0
)

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [13]:
print(tokenized_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2000
})


In [14]:
#train/validation split
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

train_dataset = train_dataset.filter(lambda x: any(label != -100 for label in x["labels"]))
eval_dataset = eval_dataset.filter(lambda x: any(label != -100 for label in x["labels"]))

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))

Filter:   0%|          | 0/1800 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

Train size: 1705
Eval size: 188


## Fine-tuning the Pre-trained Model

In [15]:
#Define the training arguments
training_args = TrainingArguments(
    output_dir="./falcon_1b_squad_finetuned",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=3e-4,
    warmup_steps=20,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    fp16=False,
    report_to="none",
)

In [16]:
#Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
)

In [17]:
trainer.train()

Step,Training Loss,Validation Loss
50,0.250705,0.221608
100,0.210449,0.209253
150,0.218375,0.192731
200,0.206735,0.183232
250,0.187898,0.182370
300,0.156576,0.175207
350,0.162666,0.176439
400,0.142679,0.168970


TrainOutput(global_step=428, training_loss=0.2591791233726751, metrics={'train_runtime': 320.9079, 'train_samples_per_second': 10.626, 'train_steps_per_second': 1.334, 'total_flos': 1.101505252294656e+16, 'train_loss': 0.2591791233726751, 'epoch': 2.0})

## Evaluating the Fine-Tuned Model

In [27]:
#question = "What happens when the immune system less active than normal?"

In [33]:
question = "What field involves the study of the immune system?"

In [34]:
context = """
    Disorders of the immune system can result in autoimmune diseases, inflammatory diseases and cancer.
    Immunodeficiency occurs when the immune system is less active than normal, resulting in recurring and life-threatening infections.
    In humans, immunodeficiency can either be the result of a genetic disease such as severe combined immunodeficiency,
    acquired conditions such as HIV/AIDS, or the use of immunosuppressive medication. In contrast, autoimmunity results from a hyperactive
    immune system attacking normal tissues as if they were foreign organisms. Common autoimmune diseases include Hashimoto's thyroiditis,
    rheumatoid arthritis, diabetes mellitus type 1, and systemic lupus erythematosus. Immunology covers the study of all aspects of the immune system.
    """

In [35]:
#Create a Prompt Template
prompt_template = """
You are an assistant for question-answering tasks. Use the following context to answer the question.
If you don't know the answer, just say that you don't know. Summarize the answer in one sentence.
Question: {question}
Context: {context}
Answer:
"""
#Create the prompt
prompt = PromptTemplate(
    input_variables=["question", "context"],
    template=prompt_template
)

In [41]:
#Create a Hugging Face pipeline
text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
    device=0 if torch.cuda.is_available() else -1
)

In [42]:
#Wrap the pipeline with HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

In [43]:
#Create and Execute the Chain
chain = prompt | llm

response = chain.invoke({
    "question": question,
    "context": context
})

print(response)


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an assistant for question-answering tasks. Use the following context to answer the question.
If you don't know the answer, just say that you don't know. Summarize the answer in one sentence.
Question: What field involves the study of the immune system?
Context: 
    Disorders of the immune system can result in autoimmune diseases, inflammatory diseases and cancer.
    Immunodeficiency occurs when the immune system is less active than normal, resulting in recurring and life-threatening infections.
    In humans, immunodeficiency can either be the result of a genetic disease such as severe combined immunodeficiency,
    acquired conditions such as HIV/AIDS, or the use of immunosuppressive medication. In contrast, autoimmunity results from a hyperactive
    immune system attacking normal tissues as if they were foreign organisms. Common autoimmune diseases include Hashimoto's thyroiditis,
    rheumatoid arthritis, diabetes mellitus type 1, and systemic lupus erythematosus. Immuno

In [44]:
import math

eval_results = trainer.evaluate()

import math
perplexity = math.exp(eval_results["eval_loss"])

print("Eval loss:", eval_results["eval_loss"])
print("Perplexity:", perplexity)

Eval loss: 0.16984935104846954
Perplexity: 1.1851262998369048
